Optimización de hiperparametros ( búsquedad en grilla o grid Search con cross validation)

Buscar la mejor combinacion de hiperparámetros de cada algoritmo

Tesis de La Plata (Forma más mecánica, software optuna). Librería de Python


In [ ]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from pathlib import Path

ROOT_DIR = Path("/content/drive/My Drive/A___Maestria_en_ID/Tareas_PLN/100_datos_thesis_maestria/Datos_SEDICI/var/")

print("Existe:", ROOT_DIR.exists())
print("Es dir:", ROOT_DIR.is_dir())

if ROOT_DIR.exists():
    # Muestra algunos elementos del primer nivel
    first_level = list(ROOT_DIR.iterdir())
    print("Items primer nivel:", len(first_level))
    print("Ejemplos:", [p.name for p in first_level[:10]])


Existe: True
Es dir: True
Items primer nivel: 1
Ejemplos: ['dspace']


In [ ]:
import os
import itertools
from pathlib import Path
import logging
from tqdm.auto import tqdm

# =========================
# CONFIG
# =========================
ROOT_DIR = Path("/content/drive/My Drive/A___Maestria_en_ID/Tareas_PLN/100_datos_thesis_maestria/Datos_SEDICI/var/")
OUTPUT_DIR = Path("/content/drive/My Drive/A___Maestria_en_ID/Tareas_PLN/100_datos_thesis_maestria/Datos_SEDICI/SEDICI_FullText_TXT/")

LOG_FILE = OUTPUT_DIR / "extract_to_txt.log"

OVERWRITE = False        # True pisa si ya existe el .txt (mismo nombre). False lo saltea.
PRECOUNT_TOTAL = False   # True = cuenta primero (2 pasadas). False = 1 pasada (sin total).

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# =========================
# LOGGING (pantalla + archivo)
# =========================
logger = logging.getLogger("extract_to_txt")
logger.setLevel(logging.INFO)
logger.handlers.clear()
logger.propagate = False  # <- evita logs duplicados en Colab

fmt = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s")

# a) Pantalla
sh = logging.StreamHandler()
sh.setLevel(logging.INFO)
sh.setFormatter(fmt)
logger.addHandler(sh)

# b) Archivo
fh = logging.FileHandler(LOG_FILE, encoding="utf-8")
fh.setLevel(logging.INFO)
fh.setFormatter(fmt)
logger.addHandler(fh)

logger.info(f"ROOT_DIR   = {ROOT_DIR}")
logger.info(f"OUTPUT_DIR = {OUTPUT_DIR}")
logger.info(f"LOG_FILE   = {LOG_FILE}")
logger.info(f"OVERWRITE  = {OVERWRITE}")
logger.info(f"PRECOUNT_TOTAL = {PRECOUNT_TOTAL}")

# =========================
# VALIDACIONES (clave para tu caso)
# =========================
if not ROOT_DIR.exists():
    raise FileNotFoundError(
        f"ROOT_DIR no existe: {ROOT_DIR}\n"
        "Tip: en Colab suele ser /content/drive/MyDrive/... si montaste con drive.mount('/content/drive')."
    )
if not ROOT_DIR.is_dir():
    raise NotADirectoryError(f"ROOT_DIR no es una carpeta: {ROOT_DIR}")

# =========================
# UTILIDADES
# =========================
def iter_files(root: Path, exclude_dir: Path):
    """Itera recursivamente todos los archivos bajo root, excluyendo la carpeta de salida."""
    root = root.resolve()
    exclude_dir = exclude_dir.resolve()

    for dirpath, dirnames, filenames in os.walk(root, topdown=True, followlinks=False):
        d = Path(dirpath).resolve()

        # Evitar entrar en OUTPUT_DIR si está dentro del ROOT_DIR
        if d == exclude_dir:
            dirnames[:] = []
            continue

        # Excluir OUTPUT_DIR si aparece como subcarpeta
        dirnames[:] = [dn for dn in dirnames if (d / dn).resolve() != exclude_dir]

        for fn in filenames:
            yield d / fn

def safe_read_text(path: Path) -> str:
    """Lee un archivo como texto de forma robusta y devuelve str."""
    data = path.read_bytes()
    try:
        return data.decode("utf-8")
    except UnicodeDecodeError:
        return data.decode("latin-1", errors="replace")

def out_txt_path(src: Path, out_dir: Path) -> Path:
    """
    Mismo nombre exacto que el original; solo extensión .txt.
    - Si no tiene extensión: agrega .txt
    - Si tiene: reemplaza la última por .txt
    """
    return out_dir / src.with_suffix(".txt").name

def count_files(root: Path, exclude_dir: Path) -> int:
    c = 0
    for _ in iter_files(root, exclude_dir):
        c += 1
    return c

# =========================
# (Opcional) smoke test: mostrar algunos archivos encontrados
# =========================
sample = list(itertools.islice(iter_files(ROOT_DIR, OUTPUT_DIR), 5))
logger.info(f"Smoke test: primeros {len(sample)} archivos encontrados: {[p.name for p in sample]}")
if len(sample) == 0:
    logger.warning("No se encontró ningún archivo bajo ROOT_DIR. Revisá que esa carpeta tenga contenido real.")

# =========================
# PROCESO
# =========================
total = count_files(ROOT_DIR, OUTPUT_DIR) if PRECOUNT_TOTAL else None
if total is not None:
    logger.info(f"Total de archivos detectados (preconteo): {total}")

detected = 0
saved = 0
skipped = 0
errors = 0

pbar = tqdm(iter_files(ROOT_DIR, OUTPUT_DIR), total=total, unit="archivo", desc="Extrayendo a TXT")

for src in pbar:
    detected += 1
    try:
        dst = out_txt_path(src, OUTPUT_DIR)

        # Si hay colisiones de nombre en carpeta plana, acá se van a pisar o saltear.
        if dst.exists() and not OVERWRITE:
            skipped += 1
            continue

        text = safe_read_text(src)
        dst.write_text(text, encoding="utf-8", errors="replace")
        saved += 1

        # Log cada 500 guardados
        if saved % 500 == 0:
            msg = f"Guardados {saved} TXT (detectados: {detected}, salteados: {skipped}, errores: {errors})"
            logger.info(msg)
            tqdm.write(msg)

        pbar.set_postfix(saved=saved, skipped=skipped, errors=errors)

    except Exception as e:
        errors += 1
        logger.exception(f"Error procesando: {src} | {e}")
        pbar.set_postfix(saved=saved, skipped=skipped, errors=errors)

logger.info("FIN")
logger.info(f"Detectados: {detected}")
logger.info(f"Guardados:  {saved}")
logger.info(f"Salteados:  {skipped}")
logger.info(f"Errores:    {errors}")
print(f"\nListo. Log: {LOG_FILE}")


2026-01-08 13:29:31,242 | INFO | ROOT_DIR   = /content/drive/My Drive/A___Maestria_en_ID/Tareas_PLN/100_datos_thesis_maestria/Datos_SEDICI/var
2026-01-08 13:29:31,256 | INFO | OUTPUT_DIR = /content/drive/My Drive/A___Maestria_en_ID/Tareas_PLN/100_datos_thesis_maestria/Datos_SEDICI/SEDICI_FullText_TXT
2026-01-08 13:29:31,265 | INFO | LOG_FILE   = /content/drive/My Drive/A___Maestria_en_ID/Tareas_PLN/100_datos_thesis_maestria/Datos_SEDICI/SEDICI_FullText_TXT/extract_to_txt.log
2026-01-08 13:29:31,270 | INFO | OVERWRITE  = False
2026-01-08 13:29:31,274 | INFO | PRECOUNT_TOTAL = False
2026-01-08 13:29:33,891 | INFO | Smoke test: primeros 5 archivos encontrados: ['61615019774315393841411321354820259456', '61614849389822153700140053766486201482', '61614835198350493707081037499233261437', '61610370582511010067090987267754478335', '61613878567949026216456117064795779187']


Extrayendo a TXT: 0archivo [00:00, ?archivo/s]

2026-01-08 13:34:03,683 | INFO | Guardados 500 TXT (detectados: 500, salteados: 0, errores: 0)


Guardados 500 TXT (detectados: 500, salteados: 0, errores: 0)


2026-01-08 13:38:28,051 | INFO | Guardados 1000 TXT (detectados: 1000, salteados: 0, errores: 0)


...

Guardados 150500 TXT (detectados: 150500, salteados: 0, errores: 0)


2026-01-09 12:36:42,389 | INFO | Guardados 151000 TXT (detectados: 151000, salteados: 0, errores: 0)


Guardados 151000 TXT (detectados: 151000, salteados: 0, errores: 0)


2026-01-09 12:41:04,700 | INFO | Guardados 151500 TXT (detectados: 151500, salteados: 0, errores: 0)


Guardados 151500 TXT (detectados: 151500, salteados: 0, errores: 0)
